# Contextual Ad Relevance Scorer
**Author:** Jessica Guo

---

## What This Project Demonstrates

This notebook builds a **contextual advertising engine** that matches ads to webpage content based on semantic similarity — without using any personal user data.

This is the core ML challenge in contextual advertising: **how do you serve relevant ads using only the content of the page, not who the user is?**

As third-party cookies become obsolete, contextual targeting is emerging as the privacy-first alternative to behavioral targeting. Companies like GumGum are leading this shift.

---

## Pipeline Overview

```
Article Content
      │
      ▼
  Preprocessing  ──────────────────────────────────
  (tokenize,                                       │
   remove stopwords)                               │
      │                                     Ad Inventory
      ▼                                            │
  TF-IDF Vectorization  ◄────────────────────────── 
  (fit on ad corpus,
   transform article)
      │
      ▼
  Cosine Similarity
  (article vector vs each ad vector)
      │
      ▼
  Ranked Ad Results
  + Explainable Keywords
```

---

## Why These Choices?

| Choice | Reason |
|--------|--------|
| **TF-IDF** | Upweights rare, domain-specific terms. 'algorithmic' matters more than 'the'. |
| **Cosine Similarity** | Length-invariant — short ads can match long articles fairly. |
| **From-scratch implementation** | Shows understanding of the math, not just API calls. |
| **Keyword explainability** | Shows which terms drove the match — critical for ad quality review. |

## 1. Imports & Setup

In [4]:
import re
import math
import json
import pandas as pd
from collections import Counter
from collections import Counter as Counter
import builtins
builtins.Counter = Counter


## 2. Sample Data

We simulate a real-world scenario: **4 articles** across different topics, and **10 ads** from various brands and categories. The scorer needs to figure out which ads belong with which articles — without any user data.

In [8]:
ARTICLES = [
    {
        'id': 'article_001',
        'title': 'The Future of Electric Vehicles: Battery Technology Breakthroughs',
        'content': '''
        Electric vehicles are rapidly transforming the automotive industry.
        Advances in lithium-ion battery technology are pushing range beyond 400 miles
        on a single charge. Major automakers including Tesla, Ford, and GM are investing
        billions in EV infrastructure and manufacturing. Charging networks are expanding
        across highways and urban centers. Government incentives and tax credits are
        making EVs more accessible to mainstream consumers. The environmental benefits
        of zero-emission vehicles are driving adoption among eco-conscious buyers.
        Battery recycling programs are addressing sustainability concerns about
        end-of-life battery disposal. Solid-state batteries promise even greater
        energy density and faster charging times in the coming years.
        '''
    },
    {
        'id': 'article_002',
        'title': 'Healthy Meal Prep: How to Eat Well on a Budget',
        'content': '''
        Meal prepping is one of the most effective strategies for maintaining a
        healthy diet while saving money. Planning your weekly meals in advance reduces
        food waste and eliminates the temptation of expensive takeout. Nutritionists
        recommend focusing on whole foods like vegetables, lean proteins, and whole grains.
        Batch cooking staples like rice, quinoa, and roasted vegetables saves time
        throughout the week. Investing in quality food storage containers keeps
        meals fresh longer. Simple recipes with minimal ingredients make the process
        sustainable long-term. Tracking macros and calories helps achieve fitness goals
        while staying within budget constraints.
        '''
    },
    {
        'id': 'article_003',
        'title': 'Machine Learning in Finance: Algorithmic Trading Strategies',
        'content': '''
        Machine learning is revolutionizing financial markets through sophisticated
        algorithmic trading systems. Hedge funds and investment banks deploy neural
        networks to identify patterns in market data and execute trades at millisecond
        speeds. Natural language processing analyzes news sentiment and earnings calls
        to predict stock movements. Risk management models use historical data to
        optimize portfolio allocation and minimize drawdown. Regulatory frameworks
        are evolving to address the challenges of AI-driven trading. Retail investors
        are gaining access to algorithmic tools through fintech platforms.
        Backtesting frameworks allow strategies to be validated against historical
        market conditions before live deployment.
        '''
    },
    {
        'id': 'article_004',
        'title': "Trail Running: A Beginner's Guide to Getting Started",
        'content': '''
        Trail running offers a refreshing alternative to road running with scenic
        routes through forests, mountains, and national parks. Beginners should
        invest in proper trail running shoes with good grip and ankle support.
        Building a base fitness level through regular running before tackling
        technical terrain is essential. Hydration packs and nutrition gels are
        important for longer trail runs. Learning to read trail maps and using
        GPS watches improves navigation on unmarked paths. Running clubs and
        local trail communities provide support and motivation for new runners.
        Recovery nutrition with protein and carbohydrates aids muscle repair
        after challenging workouts on uneven terrain.
        '''
    }
]

AD_INVENTORY = [
    {'id': 'ad_001', 'brand': 'Tesla', 'category': 'Automotive',
     'content': 'Experience the future of driving. Tesla Model 3 — 358 mile range, autopilot, zero emissions. Order yours today with federal tax credits available.'},
    {'id': 'ad_002', 'brand': 'ChargePoint', 'category': 'EV Infrastructure',
     'content': 'ChargePoint home charging station. Charge your electric vehicle overnight. Fast, reliable, smart energy management for EV owners.'},
    {'id': 'ad_003', 'brand': 'HelloFresh', 'category': 'Food & Nutrition',
     'content': 'Fresh ingredients, simple recipes, delivered to your door. HelloFresh meal kits make healthy eating easy and affordable. First box 60% off.'},
    {'id': 'ad_004', 'brand': 'MyFitnessPal', 'category': 'Health & Fitness',
     'content': 'Track calories, macros, and nutrition with MyFitnessPal. Over 14 million foods in our database. Reach your health goals faster.'},
    {'id': 'ad_005', 'brand': 'Robinhood', 'category': 'Finance',
     'content': 'Commission-free stock trading. Invest in stocks, ETFs, and crypto with Robinhood. Advanced charts, real-time data, and portfolio analytics.'},
    {'id': 'ad_006', 'brand': 'Bloomberg Terminal', 'category': 'Finance',
     'content': 'Bloomberg Terminal — real-time financial data, news analytics, and algorithmic trading tools trusted by top hedge funds and investment banks.'},
    {'id': 'ad_007', 'brand': 'Salomon', 'category': 'Outdoor Sports',
     'content': 'Salomon trail running shoes. Superior grip, lightweight design, and ankle support for technical terrain. Built for runners who push limits.'},
    {'id': 'ad_008', 'brand': 'Garmin', 'category': 'Fitness Technology',
     'content': 'Garmin Forerunner GPS watch. Track pace, elevation, heart rate, and route navigation for trail and road running. Built for endurance athletes.'},
    {'id': 'ad_009', 'brand': 'GE Appliances', 'category': 'Home',
     'content': 'GE smart refrigerators with WiFi connectivity and energy efficiency. Keep your family food fresh longer with advanced cooling technology.'},
    {'id': 'ad_010', 'brand': 'Coursera', 'category': 'Education',
     'content': 'Learn machine learning and data science online with Coursera. Courses from Stanford, Google, and top universities. Advance your tech career.'}
]

print(f'Articles loaded: {len(ARTICLES)}')
print(f'Ads loaded: {len(AD_INVENTORY)}')
print(f'Article topics: {[a["title"][:40]+"..." for a in ARTICLES]}')

Articles loaded: 4
Ads loaded: 10
Article topics: ['The Future of Electric Vehicles: Battery...', 'Healthy Meal Prep: How to Eat Well on a ...', 'Machine Learning in Finance: Algorithmic...', "Trail Running: A Beginner's Guide to Get..."]


## 3. Text Preprocessing

Before vectorizing, we clean the text:
- Lowercase everything
- Remove punctuation
- Remove stop words (common words like 'the', 'and' that carry no signal)
- Remove very short tokens

In [9]:
STOP_WORDS = {
    'a', 'an', 'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'is', 'are', 'was', 'were', 'be', 'been',
    'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
    'could', 'should', 'may', 'might', 'shall', 'can', 'this', 'that',
    'these', 'those', 'it', 'its', 'they', 'their', 'our', 'your', 'my',
    'we', 'you', 'he', 'she', 'his', 'her', 'as', 'more', 'most', 'into',
    'through', 'about', 'than', 'also', 'even', 'such', 'while', 'before',
    'after', 'through', 'across', 'between', 'among', 'how', 'what', 'which'
}

def preprocess(text: str) -> list:
    """Clean and tokenize text."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return tokens

# Test preprocessing
sample = ARTICLES[0]['content'][:200]
tokens = preprocess(sample)
print(f'Raw text: {sample[:100]}...')
print(f'Tokens: {tokens[:15]}...')
print(f'Token count: {len(tokens)}')

Raw text: 
        Electric vehicles are rapidly transforming the automotive industry.
        Advances in lit...
Tokens: ['electric', 'vehicles', 'rapidly', 'transforming', 'automotive', 'industry', 'advances', 'lithium', 'ion', 'battery', 'technology', 'pushing', 'range', 'beyond', 'miles']...
Token count: 19


## 4. TF-IDF Vectorizer (From Scratch)

**TF-IDF = Term Frequency × Inverse Document Frequency**

- **TF**: How often a term appears in THIS document
- **IDF**: How rare a term is ACROSS ALL documents (rare = more informative)
- **Result**: Common words like 'good' get downweighted. Domain-specific words like 'algorithmic' get upweighted.

$$\text{TF-IDF}(t, d) = \frac{\text{count}(t, d)}{\text{total terms in } d} \times \log\left(\frac{N+1}{df(t)+1}\right) + 1$$

In [12]:
from collections import Counter
import math

class TFIDFVectorizer:
    """
    TF-IDF vectorizer built from scratch.
    
    Why build from scratch?
    Understanding the math means you can explain every decision —
    critical when working with engineers who will scrutinize your approach.
    """
    
    def __init__(self):
        self.vocabulary = {}
        self.idf_scores = {}
        self.num_docs = 0
    
    def fit(self, documents: list) -> 'TFIDFVectorizer':
        """Learn vocabulary and IDF scores from corpus."""
        self.num_docs = len(documents)
        
        # Build vocabulary
        all_terms = set()
        for doc in documents:
            all_terms.update(doc)
        self.vocabulary = {term: idx for idx, term in enumerate(sorted(all_terms))}
        
        # Compute document frequency for each term
        doc_freq = Counter()
        for doc in documents:
            unique_terms = set(doc)
            for term in unique_terms:
                doc_freq[term] += 1
        
        # Smoothed IDF: log((N+1)/(df+1)) + 1
        # Smoothing prevents division by zero for unseen terms
        for term, df in doc_freq.items():
            self.idf_scores[term] = math.log((self.num_docs + 1) / (df + 1)) + 1
        
        return self
    
    def transform(self, document: list) -> dict:
        """Convert tokenized document to TF-IDF vector (sparse dict)."""
        tf = Counter(document)
        total_terms = len(document)
        tfidf_vector = {}
        
        for term, count in tf.items():
            if term in self.vocabulary:
                term_freq = count / total_terms
                idf = self.idf_scores.get(term, 0)
                tfidf_vector[term] = term_freq * idf
        
        return tfidf_vector
    
    def fit_transform(self, documents: list) -> list:
        """Fit on corpus and transform all documents."""
        self.fit(documents)
        return [self.transform(doc) for doc in documents]


# Test the vectorizer
vectorizer = TFIDFVectorizer()
tokenized_ads = [preprocess(ad['content']) for ad in AD_INVENTORY]
ad_vectors = vectorizer.fit_transform(tokenized_ads)

print(f'Vectorizer fitted')
print(f'Vocabulary size: {len(vectorizer.vocabulary):,} unique terms')
print(f'\nSample TF-IDF vector for "{AD_INVENTORY[0]["brand"]}" ad:')
sample_vec = sorted(ad_vectors[0].items(), key=lambda x: x[1], reverse=True)[:8]
for term, score in sample_vec:
    print(f'    {term:<20} {score:.4f}')

Vectorizer fitted
Vocabulary size: 142 unique terms

Sample TF-IDF vector for "Tesla" ad:
    experience           0.1591
    future               0.1591
    driving              0.1591
    tesla                0.1591
    model                0.1591
    mile                 0.1591
    range                0.1591
    autopilot            0.1591


## 5. Cosine Similarity

Cosine similarity measures the **angle between two vectors** in high-dimensional space.

$$\text{cosine}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

**Why cosine similarity for ads?**
- **Length-invariant**: A short ad and a long article can still match well if they share key terms
- **Interpretable**: Score of 0 = no overlap, score of 1 = identical semantic space
- **Efficient**: Can be computed in sparse form using only shared terms

In [13]:
def cosine_similarity(vec_a: dict, vec_b: dict) -> float:
    """Compute cosine similarity between two sparse TF-IDF vectors."""
    common_terms = set(vec_a.keys()) & set(vec_b.keys())
    
    if not common_terms:
        return 0.0
    
    dot_product = sum(vec_a[term] * vec_b[term] for term in common_terms)
    mag_a = math.sqrt(sum(v ** 2 for v in vec_a.values()))
    mag_b = math.sqrt(sum(v ** 2 for v in vec_b.values()))
    
    if mag_a == 0 or mag_b == 0:
        return 0.0
    
    return dot_product / (mag_a * mag_b)


# Quick sanity check — Tesla ad should score higher on EV article than food article
ev_article_tokens = preprocess(ARTICLES[0]['title'] + ' ' + ARTICLES[0]['content'])
food_article_tokens = preprocess(ARTICLES[1]['title'] + ' ' + ARTICLES[1]['content'])
ev_vec = vectorizer.transform(ev_article_tokens)
food_vec = vectorizer.transform(food_article_tokens)
tesla_vec = ad_vectors[0]  # Tesla ad

score_ev = cosine_similarity(ev_vec, tesla_vec)
score_food = cosine_similarity(food_vec, tesla_vec)

print('Sanity Check: Tesla ad vs article types')
print(f'EV article score:   {score_ev:.4f} ← should be higher')
print(f'Food article score: {score_food:.4f} ← should be lower')
print(f'Makes sense: {score_ev > score_food}')

Sanity Check: Tesla ad vs article types
EV article score:   0.3643 ← should be higher
Food article score: 0.0000 ← should be lower
Makes sense: True


## 6. Contextual Ad Scorer — Full Engine

In [15]:
from collections import Counter

class ContextualAdScorer:
    """
    Full contextual ad matching engine.
    
    Matches ads to article content using TF-IDF + Cosine Similarity.
    No user data required — purely content-based matching.
    """
    
    def __init__(self):
        self.vectorizer = TFIDFVectorizer()
        self.ad_vectors = []
        self.ads = []
        self.is_fitted = False
    
    def fit(self, ads: list) -> 'ContextualAdScorer':
        """Fit scorer on ad inventory."""
        self.ads = ads
        tokenized_ads = [preprocess(ad['content']) for ad in ads]
        self.ad_vectors = self.vectorizer.fit_transform(tokenized_ads)
        self.is_fitted = True
        print(f'Fitted on {len(ads)} ads | Vocabulary: {len(self.vectorizer.vocabulary):,} terms')
        return self
    
    def score(self, article: dict, top_k: int = 5) -> pd.DataFrame:
        """Score all ads against an article, return top_k with explanations."""
        if not self.is_fitted:
            raise RuntimeError('Call fit() before score()')
        
        article_text = article['title'] + ' ' + article['content']
        article_tokens = preprocess(article_text)
        article_vector = self.vectorizer.transform(article_tokens)
        
        results = []
        for ad, ad_vector in zip(self.ads, self.ad_vectors):
            score = cosine_similarity(article_vector, ad_vector)
            
            # Explainability: which shared terms drove the match?
            common_terms = set(article_vector.keys()) & set(ad_vector.keys())
            top_keywords = sorted(
                common_terms,
                key=lambda t: article_vector.get(t, 0) * ad_vector.get(t, 0),
                reverse=True
            )[:5]
            
            results.append({
                'ad_id': ad['id'],
                'brand': ad['brand'],
                'category': ad['category'],
                'relevance_score': round(score, 4),
                'top_matching_keywords': ', '.join(top_keywords) if top_keywords else 'none',
                'ad_preview': ad['content'][:80] + '...'
            })
        
        results_df = (pd.DataFrame(results)
                      .sort_values('relevance_score', ascending=False)
                      .head(top_k)
                      .reset_index(drop=True))
        results_df.index = results_df.index + 1
        results_df.index.name = 'rank'
        
        return results_df


# Initialize and fit
scorer = ContextualAdScorer()
scorer.fit(AD_INVENTORY)

Fitted on 10 ads | Vocabulary: 142 terms


## 7. Score Articles — Results

In [16]:
# Score each article and display results
all_results = {}

for article in ARTICLES:
    print(f"\n{'='*65}")
    print(f"{article['title']}")
    print('='*65)
    
    results = scorer.score(article, top_k=5)
    all_results[article['id']] = {
        'article_title': article['title'],
        'top_ads': results
    }
    
    display_cols = ['brand', 'category', 'relevance_score', 'top_matching_keywords']
    print(results[display_cols].to_string())


The Future of Electric Vehicles: Battery Technology Breakthroughs
              brand           category  relevance_score               top_matching_keywords
rank                                                                                       
1             Tesla         Automotive           0.3643   range, future, driving, tax, zero
2       ChargePoint  EV Infrastructure           0.3349  electric, charging, charge, energy
3     GE Appliances               Home           0.1627                  technology, energy
4      MyFitnessPal   Health & Fitness           0.0602                              faster
5        HelloFresh   Food & Nutrition           0.0000                                none

Healthy Meal Prep: How to Eat Well on a Budget
              brand          category  relevance_score                        top_matching_keywords
rank                                                                                               
1        HelloFresh  Food & Nutrition    

## 8. Evaluation Metrics

Key metrics to assess the quality of our contextual matching:

- **Top Score**: Confidence in the best match
- **Avg Top-3 Score**: Overall quality of recommendations
- **Discrimination Gap**: Difference between #1 and #3 — higher gap = model is more decisive

In [19]:
eval_rows = []
for article_id, data in all_results.items():
    top_ads = data['top_ads']
    eval_rows.append({
        'article': data['article_title'][:45] + '...',
        'top_match': top_ads.iloc[0]['brand'],
        'top_score': top_ads.iloc[0]['relevance_score'],
        'avg_top3': round(top_ads.head(3)['relevance_score'].mean(), 4),
        'discrimination_gap': round(
            top_ads.iloc[0]['relevance_score'] - top_ads.iloc[2]['relevance_score'], 4
        )
    })

eval_df = pd.DataFrame(eval_rows)
print('Evaluation Summary')
print('='*80)
print(eval_df.to_string(index=False))
print(f'\nAverage top score across articles: {eval_df["top_score"].mean():.4f}')
print(f'Average discrimination gap: {eval_df["discrimination_gap"].mean():.4f}')

Evaluation Summary
                                         article          top_match  top_score  avg_top3  discrimination_gap
The Future of Electric Vehicles: Battery Tech...              Tesla     0.3643    0.2873              0.2016
Healthy Meal Prep: How to Eat Well on a Budge...         HelloFresh     0.4405    0.2954              0.2264
Machine Learning in Finance: Algorithmic Trad... Bloomberg Terminal     0.5703    0.3436              0.3442
Trail Running: A Beginner's Guide to Getting ...            Salomon     0.5074    0.3034              0.4427

Average top score across articles: 0.4706
Average discrimination gap: 0.3037


## 9. Score Distribution Visualization

Let's visualize how scores are distributed across all articles to understand model behavior.

In [23]:
# Score matrix: all articles vs all ads
article_titles = [a['title'][:35] + '...' for a in ARTICLES]
ad_brands = [ad['brand'] for ad in AD_INVENTORY]

score_matrix = []
for article in ARTICLES:
    article_text = article['title'] + ' ' + article['content']
    article_tokens = preprocess(article_text)
    article_vector = scorer.vectorizer.transform(article_tokens)
    
    row = []
    for ad_vector in scorer.ad_vectors:
        score = cosine_similarity(article_vector, ad_vector)
        row.append(round(score, 4))
    score_matrix.append(row)

score_df = pd.DataFrame(score_matrix, index=article_titles, columns=ad_brands)

print('Full Relevance Score Matrix (Articles × Ads)')
print('Higher score = more contextually relevant')
print('='*90)
print(score_df.to_string())
print('\nModel correctly identifies domain-specific ad-article pairs')

Full Relevance Score Matrix (Articles × Ads)
Higher score = more contextually relevant
                                         Tesla  ChargePoint  HelloFresh  MyFitnessPal  Robinhood  Bloomberg Terminal  Salomon  Garmin  GE Appliances  Coursera
The Future of Electric Vehicles: Ba...  0.3643       0.3349      0.0000        0.0602     0.0000              0.0000   0.0000   0.000         0.1627    0.0000
Healthy Meal Prep: How to Eat Well ...  0.0000       0.0000      0.4405        0.2316     0.0397              0.0383   0.0000   0.000         0.2141    0.0547
Machine Learning in Finance: Algori...  0.0000       0.0456      0.0000        0.0000     0.2345              0.5703   0.0000   0.000         0.0000    0.2261
Trail Running: A Beginner's Guide t...  0.0000       0.0000      0.0000        0.0647     0.0307              0.0000   0.5074   0.338         0.0322    0.0306

Model correctly identifies domain-specific ad-article pairs


## 10. Model Limitations & Production Extensions

### Current Limitations

| Limitation | Impact |
|-----------|--------|
| TF-IDF misses semantic meaning | 'car' and 'automobile' scored separately |
| No synonym handling | Vocabulary-dependent matching |
| Static ad inventory | No real-time ad updates |
| No click feedback loop | Can't learn from user behavior |

### Production Extensions

In a real contextual advertising system at scale, this pipeline would extend to:

1. **Transformer embeddings** (BERT, sentence-transformers) for deeper semantic understanding — 'car' and 'automobile' would have similar vectors

2. **Real-time inference** with sub-100ms latency for RTB auction integration — every ad impression triggers a model call in milliseconds

3. **Distributed processing** with Apache Spark for billions of daily page impressions — TF-IDF vectors computed in parallel across a cluster

4. **A/B testing framework** to compare contextual models against behavioral targeting baselines — measure CTR, viewability, and attention metrics

5. **MLOps pipeline** with drift monitoring, automated retraining, and model versioning via MLflow

6. **Brand safety layer** — ensure ads don't appear next to content that conflicts with brand values

In [24]:
# Save results
json_output = {}
for article_id, data in all_results.items():
    json_output[article_id] = {
        'article_title': data['article_title'],
        'top_ads': data['top_ads'].reset_index().to_dict(orient='records')
    }

with open('results.json', 'w') as f:
    json.dump(json_output, f, indent=2)

print('Results saved to results.json')
print('\nProject complete!')
print('\nKey takeaways:')
print('  • TF-IDF + cosine similarity is a strong, interpretable baseline for contextual matching')
print('  • Keyword explainability is critical for ad quality review in production')
print('  • The discrimination gap tells you how decisive the model is — important for ranking')
print('  • Zero user data required — fully privacy-compliant contextual targeting')

Results saved to results.json

Project complete!

Key takeaways:
  • TF-IDF + cosine similarity is a strong, interpretable baseline for contextual matching
  • Keyword explainability is critical for ad quality review in production
  • The discrimination gap tells you how decisive the model is — important for ranking
  • Zero user data required — fully privacy-compliant contextual targeting
